# NYSC Survival · real streets and landmarks from OpenStreetMap
Run all cells (Runtime → Run all). It takes about 15–25 minutes, needs no key and costs nothing.
At the end it downloads `nysc_osm.zip` → send it to Claude.

For each state it takes a ~700 m circle around a real town centre (roads, building footprints, shops, bus stops, names)
plus the famous places within 20 km (anything OpenStreetMap links to Wikidata).
Data © OpenStreetMap contributors, ODbL. The game credits it.

In [ ]:
import json, math, time, os, requests, shutil
AREAS = [["AB", "Umuahia, Abia State"], ["AD", "Jimeta, Yola, Adamawa State"], ["AK", "Uyo, Akwa Ibom State"], ["AN", "Awka, Anambra State"], ["BA", "Bauchi, Bauchi State"], ["BY", "Yenagoa, Bayelsa State"], ["BE", "Makurdi, Benue State"], ["BO", "Maiduguri, Borno State"], ["CR", "Calabar, Cross River State"], ["DT", "Asaba, Delta State"], ["EB", "Abakaliki, Ebonyi State"], ["ED", "Ring Road, Benin City, Edo State"], ["EK", "Ado-Ekiti, Ekiti State"], ["EN", "Okpara Square, Enugu, Enugu State"], ["FC", "Wuse, Abuja, FCT"], ["GB", "Gombe, Gombe State"], ["IM", "Owerri, Imo State"], ["JG", "Dutse, Jigawa State"], ["KD", "Kaduna, Kaduna State"], ["KN", "Kano, Kano State"], ["KT", "Katsina, Katsina State"], ["KB", "Birnin Kebbi, Kebbi State"], ["KG", "Lokoja, Kogi State"], ["KW", "Ilorin, Kwara State"], ["LA", "Ikeja, Lagos State"], ["NR", "Lafia, Nasarawa State"], ["NG", "Minna, Niger State"], ["OG", "Abeokuta, Ogun State"], ["OD", "Akure, Ondo State"], ["OS", "Osogbo, Osun State"], ["OY", "Dugbe, Ibadan, Oyo State"], ["PL", "Jos, Plateau State"], ["RV", "Port Harcourt, Rivers State"], ["SO", "Sokoto, Sokoto State"], ["TR", "Jalingo, Taraba State"], ["YB", "Damaturu, Yobe State"], ["ZA", "Gusau, Zamfara State"]]
UA = {'User-Agent': 'NYSC-Survival-game/1.0 (github.com/Auspicious14/nysc-survival)'}
OVERPASS = ['https://overpass-api.de/api/interpreter', 'https://overpass.kumi.systems/api/interpreter', 'https://overpass.private.coffee/api/interpreter']
OUT = '/content/nysc_osm'; os.makedirs(OUT, exist_ok=True)
R, RL = 700, 20000
def geocode(q):
    r = requests.get('https://nominatim.openstreetmap.org/search', params={'q': q + ', Nigeria', 'format': 'json', 'limit': 1}, headers=UA, timeout=60)
    time.sleep(1.2); j = r.json()
    return (float(j[0]['lat']), float(j[0]['lon'])) if j else None
def overpass(q):
    for attempt in range(6):
        url = OVERPASS[attempt % len(OVERPASS)]
        try:
            r = requests.post(url, data={'data': q}, headers=UA, timeout=240)
            if r.status_code == 200: return r.json()
            print('   overpass', r.status_code, 'on', url.split('/')[2], '→ retry'); time.sleep(20)
        except Exception as e: print('   overpass error', str(e)[:80]); time.sleep(20)
    raise RuntimeError('overpass failed')
print('ok')

In [ ]:
def local(lat0, lon0):
    k = 111320 * math.cos(math.radians(lat0))
    return lambda lat, lon: (round((lon - lon0) * k, 1), round((lat - lat0) * 111320, 1))
KEEP_ROAD = {'motorway','trunk','primary','secondary','tertiary','unclassified','residential','service','living_street','pedestrian','footway','path','track','motorway_link','trunk_link','primary_link','secondary_link','tertiary_link'}
def area(code_, q):
    c = geocode(q)
    if not c: print(code_, 'geocode failed'); return
    lat, lon = c; P = local(lat, lon)
    a = f'(around:{R},{lat},{lon})'
    j = overpass(f'''[out:json][timeout:200];
(way["highway"]{a}; way["building"]{a}; way["landuse"]{a}; way["leisure"]{a}; way["natural"]{a}; way["waterway"]{a}; way["amenity"]{a};
 node["amenity"]["name"]{a}; node["shop"]{a}; node["highway"="bus_stop"]{a}; node["office"]["name"]{a}; node["tourism"]{a}; node["place"]{a};);
out tags geom;''')
    roads, bld, poi, green, water = [], [], [], [], []
    for e in j['elements']:
        t = e.get('tags', {}); nm = t.get('name', '')
        if e['type'] == 'node':
            k = t.get('amenity') or t.get('shop') or t.get('office') or t.get('tourism') or ('bus_stop' if t.get('highway') == 'bus_stop' else '') or ('place:' + t['place'] if 'place' in t else '')
            if k: poi.append([k, nm, *P(e['lat'], e['lon'])])
            continue
        g = e.get('geometry')
        if not g: continue
        pts = [v for p in g for v in P(p['lat'], p['lon'])]
        if 'highway' in t and t['highway'] in KEEP_ROAD:
            roads.append([t['highway'], nm, int(t.get('lanes', '0') if str(t.get('lanes', '0')).isdigit() else 0), 1 if t.get('oneway') == 'yes' else 0, pts])
        elif 'building' in t:
            lv = t.get('building:levels', ''); lv = int(float(lv)) if lv.replace('.', '', 1).isdigit() else 0
            bld.append([t['building'], nm, lv, t.get('amenity') or t.get('shop') or t.get('religion') or '', pts])
        elif t.get('natural') == 'water' or 'waterway' in t:
            water.append([t.get('waterway') or 'water', nm, pts])
        elif t.get('landuse') in ('grass', 'forest', 'meadow', 'farmland', 'recreation_ground', 'cemetery') or t.get('leisure') in ('park', 'pitch', 'garden', 'stadium') or t.get('natural') in ('wood', 'scrub', 'grassland'):
            green.append([t.get('leisure') or t.get('landuse') or t.get('natural'), nm, pts])
        elif 'amenity' in t:
            poi.append([t['amenity'], nm, *P(*map(lambda v: sum(v) / len(v), zip(*[(p['lat'], p['lon']) for p in g])))])
    # notable places within 20 km (linked to Wikidata)
    a2 = f'(around:{RL},{lat},{lon})'
    j2 = overpass(f'''[out:json][timeout:200];
(nwr["wikidata"]["name"]["tourism"]{a2}; nwr["wikidata"]["name"]["historic"]{a2}; nwr["wikidata"]["name"]["amenity"="place_of_worship"]{a2};
 nwr["wikidata"]["name"]["leisure"="stadium"]{a2}; nwr["wikidata"]["name"]["man_made"]{a2}; nwr["wikidata"]["name"]["natural"]{a2};
 nwr["wikidata"]["name"]["bridge"]{a2}; nwr["wikidata"]["name"]["amenity"="university"]{a2}; nwr["wikidata"]["name"]["building"]{a2}; nwr["wikidata"]["name"]["aeroway"="aerodrome"]{a2};
 nwr["wikidata"]["name"]["amenity"="marketplace"]{a2}; nwr["wikidata"]["name"]["office"="government"]{a2};);
out tags center;''')
    lm = []
    for e in j2['elements']:
        t = e['tags']; cc = e.get('center') or {'lat': e.get('lat'), 'lon': e.get('lon')}
        if cc['lat'] is None: continue
        x, y = P(cc['lat'], cc['lon'])
        kind = next((f'{k}={t[k]}' for k in ('tourism', 'historic', 'amenity', 'leisure', 'man_made', 'natural', 'bridge', 'aeroway', 'office', 'building') if k in t), '')
        lm.append([kind, t['name'], round(math.hypot(x, y)), round(math.degrees(math.atan2(x, y)) % 360), t.get('religion', ''), t.get('ele', ''), t.get('height', ''), t.get('wikidata', ''), x, y])
    out = {'id': code_, 'q': q, 'center': [lat, lon], 'r': R, 'roads': roads, 'bld': bld, 'poi': poi, 'green': green, 'water': water, 'landmarks': sorted(lm, key=lambda v: v[2])}
    json.dump(out, open(f'{OUT}/{code_}.json', 'w'), separators=(',', ':'), ensure_ascii=False)
    print(f"{code_} {q}: {len(roads)} roads · {len(bld)} buildings · {len(poi)} places · {len(lm)} landmarks · {os.path.getsize(f'{OUT}/{code_}.json') // 1024} KB")
for code_, q in AREAS:
    if os.path.exists(f'{OUT}/{code_}.json'): continue
    try: area(code_, q)
    except Exception as e: print(code_, 'FAILED', e)
    time.sleep(5)
print('done — run the next cell')

In [ ]:
# rerun the cell above first if any state says FAILED (it skips states already done)
shutil.make_archive('/content/nysc_osm', 'zip', OUT)
print(len(os.listdir(OUT)), 'states saved')
from google.colab import files; files.download('/content/nysc_osm.zip')